# Reproducing the PGM'26 results

This notebook mirrors `script/reproduce_PGM_paper.py` and explains each step. The full experiments (11 datasets, 2 seeds, 5 runs each) take a long time, so by default this notebook runs a **quick configuration** on three datasets. Set `QUICK = False` to run the full paper configuration (writes to `results/pgm26`).

The pipeline consists of three scripts in `experiments/pgm26/`:

| step | script | output |
|---|---|---|
| 1 | `run_experiments.py` | raw metrics (`csv/`), accuracy-rejection curves (`accaccs/`) and AUCs (`auc_summary/`) |
| 2 | `make_arc_csvs.py` | averaged ARCs used for the ARC figures |
| 3 | `make_rr_tables.py` | accuracy and number of wins per rejection rate, and AU-ARC (Tables 3 and 4) |

In [ ]:
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
QUICK = True
RESULTS = REPO_ROOT / 'results' / ('pgm26_quick' if QUICK else 'pgm26')

## Run the experiments and post-process the results

In [ ]:
cmd = [sys.executable, str(REPO_ROOT / 'script' / 'reproduce_PGM_paper.py')]
if QUICK:
    cmd.append('--quick')
subprocess.run(cmd, check=True, capture_output=True)
print('results written to', RESULTS)

## AUC of every reliability metric

For each dataset the metrics are sorted by the area under their accuracy-rejection curve (mean ± standard deviation over the random train/test splits).

In [ ]:
print((RESULTS / 'auc_summary' / '0_NBC.dat').read_text())

## Accuracy-rejection curves

Curves averaged over the seeds, as plotted in the paper (`latex_plots/ARCs`). Here only the plots for the first dataset are shown.

In [ ]:
arc_dir = RESULTS / 'latex_plots' / 'ARCs' / 'csvs'
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, model in zip(axes, ['NBC', 'GeFs']):
    data = np.genfromtxt(arc_dir / f'Australian_ALL_{model}_arc.csv', delimiter=',', names=True)
    for metric in data.dtype.names[1:]:
        ax.plot(data['n'], data[metric], label=metric)
    ax.set_title(f'Australian - {model}')
    ax.set_xlabel('number of rejected instances')
axes[0].set_ylabel('accuracy')
axes[1].legend(fontsize=7)
plt.tight_layout()
plt.show()

## Accuracy and wins per rejection rate

For every rejection rate: the mean accuracy (%) over all datasets of each metric, and the number of datasets (W) on which the metric is the best. The last row contains the mean AU-ARC. This is the csv that is typeset by `latex_plots/RR_ARC/stalone_rr_combi_*.tex`.

In [ ]:
# do not print last line of table_rr_combi_NBC.csv
print((RESULTS / 'latex_plots' / 'RR_ARC' / 'csvs' / 'table_rr_combi_NBC.csv').read_text().rsplit('\n',2)[0])
